# 📦 구매 데이터 AX 실습
### 엑셀 수작업 → 자동화 파이프라인 + AI 수요 예측

**아서당 13기 달료**

---

## 실습 목표

| 단계 | 내용 | 해결하는 문제 |
|------|------|---------------|
| 1 | 샘플 구매 데이터 생성 | 엑셀 기반 데이터 시뮬레이션 |
| 2 | 데이터 파이프라인 자동화 | 수작업 정리 → 자동화 |
| 3 | EDA + 시각화 | 패턴/계절성 파악 |
| 4 | 수요 예측 모델 3종 | 경험 의존 → 모델 기반 전환 |
| 5 | 모델 성능 평가 (MAPE) | 예측 정확도 정량화 |
| 6 | LLM 리포트 자동 생성 | 인사이트 요약 자동화 |

---
## 공통 AX 프레임워크
```
[Input: 엑셀 구매 데이터]
    → [ETL 자동화]
    → [Forecasting Model (MA / Holt-Winters / XGBoost)]
    → [Output: 예측값 + 자동 리포트]
    → [Feedback Loop]
```

## 0. 환경 설정

In [ ]:
# 필요한 패키지 설치 (Colab 또는 로컬 최초 실행 시)
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels openpyxl openai -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from xgboost import XGBRegressor
import warnings
import os

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'   # macOS
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'NanumGothic'   # Linux/Colab
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
print('✅ 환경 설정 완료')

---
## 1. 샘플 구매 데이터 생성

> **현실 상황**: 구매 담당자가 매월 엑셀에 직접 입력하는 데이터를 시뮬레이션합니다.

**가상 회사**: 중견 유통사 / 구매 품목 6종 / 2년치 월별 데이터

In [ ]:
# ── 상품 마스터 정의 ──────────────────────────────────────────
products = {
    'P001': {'name': '생수 (2L, 박스)', 'category': '음료/식품', 'unit_price': 12000,  'base_demand': 800},
    'P002': {'name': '커피믹스 (100입)', 'category': '음료/식품', 'unit_price': 18500,  'base_demand': 450},
    'P003': {'name': 'A4 용지 (박스)',   'category': '사무용품',  'unit_price': 22000,  'base_demand': 300},
    'P004': {'name': '청소용품 세트',    'category': '소모품',    'unit_price': 35000,  'base_demand': 180},
    'P005': {'name': '마스크 (50매)',    'category': '소모품',    'unit_price': 9800,   'base_demand': 250},
    'P006': {'name': '손소독제 (500ml)','category': '소모품',    'unit_price': 7500,   'base_demand': 200},
}

# ── 계절성 가중치 (월별 1~12) ────────────────────────────────
seasonality = {
    'P001': [0.9, 0.8, 0.9, 1.0, 1.1, 1.3, 1.5, 1.5, 1.1, 1.0, 0.9, 0.8],   # 여름 피크
    'P002': [1.2, 1.1, 1.0, 0.9, 0.9, 0.8, 0.8, 0.8, 1.0, 1.1, 1.2, 1.3],   # 겨울 피크
    'P003': [1.2, 1.0, 1.3, 1.0, 0.9, 0.8, 0.7, 0.8, 1.2, 1.1, 1.1, 1.2],   # 학기 초 피크
    'P004': [0.9, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 0.9],   # 평탄
    'P005': [1.3, 1.2, 1.1, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2, 1.3, 1.4],   # 동절기 피크
    'P006': [1.1, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2, 1.2, 1.1, 1.0, 1.0, 1.1],   # 여름 소폭 상승
}

# ── 데이터 생성 (3년치: Holt-Winters 최소 2 seasonal cycle 확보) ──
records = []
dates = pd.date_range('2022-01-01', '2024-12-01', freq='MS')  # 36개월

for date in dates:
    for pid, meta in products.items():
        month_idx = date.month - 1
        trend = 1.0 + (date.year - 2022) * 0.08  # 연 8% 성장
        noise = np.random.normal(1.0, 0.07)       # ±7% 노이즈

        demand = int(meta['base_demand'] * seasonality[pid][month_idx] * trend * noise)
        stock_in = int(demand * np.random.uniform(0.95, 1.10))
        opening_stock = int(demand * np.random.uniform(0.10, 0.25))
        closing_stock = max(0, opening_stock + stock_in - demand)

        records.append({
            '날짜':       date,
            '상품코드':   pid,
            '상품명':     meta['name'],
            '카테고리':   meta['category'],
            '발주량':     stock_in,
            '입고량':     stock_in,
            '판매량':     demand,
            '기초재고':   opening_stock,
            '기말재고':   closing_stock,
            '단가':       meta['unit_price'],
            '매출':       demand * meta['unit_price'],
            '발주금액':   stock_in * meta['unit_price'],
        })

df = pd.DataFrame(records)
print(f'✅ 데이터 생성 완료: {len(df)}행 × {len(df.columns)}열')
df.head(12)

In [ ]:
# 엑셀 파일로 저장 (실무 상황 시뮬레이션)
df.to_excel('purchase_data_sample.xlsx', index=False)
print('✅ purchase_data_sample.xlsx 저장 완료')

# 기본 통계
print('\n📊 기본 통계')
print(f'  기간: {df["날짜"].min().strftime("%Y-%m")} ~ {df["날짜"].max().strftime("%Y-%m")}')
print(f'  총 매출: {df["매출"].sum():,.0f}원')
print(f'  총 발주금액: {df["발주금액"].sum():,.0f}원')
print(f'  상품 수: {df["상품코드"].nunique()}개')
print(f'  카테고리 수: {df["카테고리"].nunique()}개')

---
## 2. 데이터 파이프라인 자동화

> **해결하는 문제**: 매월 엑셀을 열어 수작업으로 정리하던 것을 코드 한 줄로 처리

In [ ]:
# ── Step 1: 엑셀 로드 ────────────────────────────────────────
raw = pd.read_excel('purchase_data_sample.xlsx', parse_dates=['날짜'])

# ── Step 2: 자동 검증 (데이터 품질 체크) ─────────────────────
def validate_data(df):
    issues = []
    if df.isnull().sum().sum() > 0:
        issues.append(f'결측값: {df.isnull().sum().sum()}개')
    neg_rows = (df[['판매량', '발주량', '기말재고']] < 0).any(axis=1).sum()
    if neg_rows:
        issues.append(f'음수값 행: {neg_rows}개')
    dup_rows = df.duplicated(['날짜', '상품코드']).sum()
    if dup_rows:
        issues.append(f'중복 행: {dup_rows}개')
    if issues:
        print('⚠️  데이터 이슈 발견:')
        for i in issues: print(f'  - {i}')
    else:
        print('✅ 데이터 검증 통과')
    return df

raw = validate_data(raw)

# ── Step 3: 파생 변수 생성 ────────────────────────────────────
raw['연도']      = raw['날짜'].dt.year
raw['월']        = raw['날짜'].dt.month
raw['재고회전율'] = (raw['판매량'] / (raw['기초재고'] + 1)).round(2)  # +1 division guard
raw['마진율']    = 0.35  # 가정: 35% 마진
raw['이익']      = (raw['매출'] * raw['마진율']).astype(int)

# ── Step 4: 자동 월별 요약 리포트 ────────────────────────────
monthly_report = (
    raw.groupby('날짜')
    .agg(
        총발주량=('발주량', 'sum'),
        총판매량=('판매량', 'sum'),
        총매출=('매출', 'sum'),
        총발주금액=('발주금액', 'sum'),
        평균재고회전율=('재고회전율', 'mean'),
        품목수=('상품코드', 'nunique'),
    )
    .reset_index()
)
monthly_report['월매출증감률(%)'] = monthly_report['총매출'].pct_change() * 100

print('\n📋 자동 생성된 월별 요약 리포트')
monthly_report.style.format({
    '총매출': '{:,.0f}',
    '총발주금액': '{:,.0f}',
    '평균재고회전율': '{:.2f}',
    '월매출증감률(%)': '{:+.1f}%'
})

In [ ]:
# ── Step 5: 리포트 엑셀 자동 저장 ────────────────────────────
with pd.ExcelWriter('purchase_report_auto.xlsx', engine='openpyxl') as writer:
    raw.to_excel(writer, sheet_name='원본데이터', index=False)
    monthly_report.to_excel(writer, sheet_name='월별요약', index=False)
    raw.groupby(['카테고리', '날짜'])['매출'].sum().unstack().to_excel(writer, sheet_name='카테고리별매출')

print('✅ purchase_report_auto.xlsx 자동 생성 완료 (3개 시트)')
print('   → 원본데이터 / 월별요약 / 카테고리별매출')

---
## 3. EDA: 탐색적 데이터 분석 + 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('구매 데이터 EDA 대시보드', fontsize=16, fontweight='bold', y=1.01)

# ── (1) 월별 총매출 추이 ──────────────────────────────────────
ax = axes[0, 0]
ax.plot(monthly_report['날짜'], monthly_report['총매출'] / 1e6, color='#4F46E5', linewidth=2.5, marker='o', markersize=4)
ax.fill_between(monthly_report['날짜'], monthly_report['총매출'] / 1e6, alpha=0.15, color='#4F46E5')
ax.set_title('월별 총매출 추이', fontweight='bold')
ax.set_ylabel('매출 (백만원)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=30)

# ── (2) 카테고리별 매출 비중 ──────────────────────────────────
ax = axes[0, 1]
cat_sales = raw.groupby('카테고리')['매출'].sum()
colors = ['#4F46E5', '#10B981', '#F59E0B']
wedges, texts, autotexts = ax.pie(
    cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts: at.set_fontsize(10)
ax.set_title('카테고리별 매출 비중', fontweight='bold')

# ── (3) 상품별 월 평균 판매량 히트맵 ─────────────────────────
ax = axes[1, 0]
heatmap_data = raw.pivot_table(values='판매량', index='상품명', columns='월', aggfunc='mean')
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd', fmt='.0f', annot=True,
            linewidths=0.5, cbar_kws={'label': '월 평균 판매량'})
ax.set_title('상품 × 월 판매량 히트맵 (계절성)', fontweight='bold')
ax.set_xlabel('월')
ax.tick_params(axis='y', rotation=0)

# ── (4) 월별 발주금액 vs 매출 비교 ───────────────────────────
ax = axes[1, 1]
x = np.arange(len(monthly_report))
w = 0.35
ax.bar(x - w/2, monthly_report['총발주금액'] / 1e6, w, label='발주금액', color='#F59E0B', alpha=0.8)
ax.bar(x + w/2, monthly_report['총매출'] / 1e6, w, label='매출', color='#4F46E5', alpha=0.8)
ax.set_title('월별 발주금액 vs 매출', fontweight='bold')
ax.set_ylabel('금액 (백만원)')
ax.set_xticks(x[::2])
ax.set_xticklabels([d.strftime('%y.%m') for d in monthly_report['날짜'].iloc[::2]], rotation=30)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA 대시보드 저장: eda_dashboard.png')

In [ ]:
# ── 계절성 분해 (가장 많이 팔리는 상품) ─────────────────────
top_product = raw.groupby('상품명')['판매량'].sum().idxmax()
ts_data = raw[raw['상품명'] == top_product].set_index('날짜')['판매량'].sort_index()

decomp = seasonal_decompose(ts_data, model='multiplicative', period=12, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle(f'계절성 분해: {top_product}', fontsize=14, fontweight='bold')
components = [
    (decomp.observed,  '실제 판매량',  '#4F46E5'),
    (decomp.trend,     '추세 (Trend)', '#10B981'),
    (decomp.seasonal,  '계절성 (Seasonality)', '#F59E0B'),
    (decomp.resid,     '잔차 (Residual)', '#EF4444'),
]
for ax, (data, title, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
print(f'\n💡 인사이트: {top_product}은(는) 여름(7~8월)에 약 {decomp.seasonal.max():.1f}배 수요 급증')

---
## 4. 수요 예측 모델 3종 비교

| 모델 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Moving Average** | 과거 N개월 평균 | 단순, 직관적 | 계절성 반영 불가 |
| **Holt-Winters** | 추세 + 계절성 감안 지수평활 | 계절성 자동 반영 | 급격한 변화에 약함 |
| **XGBoost** | ML 기반 피처 엔지니어링 | 복잡한 패턴 포착 | 데이터 많을수록 유리 |

> **목표**: 2024년 11~12월 2개월 예측 후 실제값과 비교 → MAPE 산출

In [ ]:
# ── 예측 대상: 생수 (전체 판매량 합산) ──────────────────────
TARGET = 'P001'  # 생수 (2L, 박스)
target_name = products[TARGET]['name']

ts = (
    raw[raw['상품코드'] == TARGET]
    .set_index('날짜')['판매량']
    .sort_index()
)

# Train: 2022-01 ~ 2024-10 (34개월) / Test: 2024-11 ~ 2024-12 (2개월)
train = ts[:'2024-10']
test  = ts['2024-11':]

print(f'예측 대상: {target_name}')
print(f'학습 기간: {train.index[0].strftime("%Y-%m")} ~ {train.index[-1].strftime("%Y-%m")} ({len(train)}개월)')
print(f'테스트 기간: {test.index[0].strftime("%Y-%m")} ~ {test.index[-1].strftime("%Y-%m")} ({len(test)}개월)')

In [ ]:
# ── Model 1: Moving Average (3개월) ─────────────────────────
window = 3
ma_pred = train.rolling(window=window).mean().iloc[-1]  # 마지막 3개월 평균
ma_forecast = pd.Series([ma_pred] * len(test), index=test.index)

ma_mape = mean_absolute_percentage_error(test, ma_forecast) * 100
ma_mae  = mean_absolute_error(test, ma_forecast)

print(f'📊 Model 1 | Moving Average ({window}개월)')
print(f'   예측값: {ma_forecast.values}')
print(f'   실제값: {test.values}')
print(f'   MAPE: {ma_mape:.1f}% | MAE: {ma_mae:.0f}개')

In [ ]:
# ── Model 2: Holt-Winters (추세 + 계절성) ────────────────────
hw_model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='mul',
    seasonal_periods=12
).fit(optimized=True)

hw_forecast = hw_model.forecast(len(test))
hw_forecast.index = test.index

hw_mape = mean_absolute_percentage_error(test, hw_forecast) * 100
hw_mae  = mean_absolute_error(test, hw_forecast)

print(f'📊 Model 2 | Holt-Winters')
print(f'   예측값: {hw_forecast.values.round(0)}')
print(f'   실제값: {test.values}')
print(f'   MAPE: {hw_mape:.1f}% | MAE: {hw_mae:.0f}개')

In [ ]:
# ── Model 3: XGBoost (피처 엔지니어링) ───────────────────────
def make_features(series: pd.Series) -> pd.DataFrame:
    df = series.to_frame(name='y')
    df['month']      = df.index.month
    df['quarter']    = df.index.quarter
    df['lag_1']      = df['y'].shift(1)
    df['lag_3']      = df['y'].shift(3)
    df['lag_12']     = df['y'].shift(12)
    df['rolling_3']  = df['y'].shift(1).rolling(3).mean()
    df['rolling_6']  = df['y'].shift(1).rolling(6).mean()
    return df.dropna()

# 전체 데이터로 피처 생성 후 train/test 분리
feat_df = make_features(ts)
X_cols  = [c for c in feat_df.columns if c != 'y']

X_train = feat_df.loc[:'2024-10', X_cols]
y_train = feat_df.loc[:'2024-10', 'y']
X_test  = feat_df.loc['2024-11':, X_cols]
y_test  = feat_df.loc['2024-11':, 'y']

xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)

xgb_forecast_vals = xgb_model.predict(X_test)
xgb_forecast = pd.Series(xgb_forecast_vals, index=y_test.index)

xgb_mape = mean_absolute_percentage_error(y_test, xgb_forecast) * 100
xgb_mae  = mean_absolute_error(y_test, xgb_forecast)

print(f'📊 Model 3 | XGBoost')
print(f'   예측값: {xgb_forecast.values.round(0)}')
print(f'   실제값: {y_test.values}')
print(f'   MAPE: {xgb_mape:.1f}% | MAE: {xgb_mae:.0f}개')

---
## 5. 모델 성능 비교 (MAPE 기준)

In [ ]:
results = pd.DataFrame([
    {'모델': f'Moving Average ({window}M)', 'MAPE(%)': ma_mape,  'MAE(개)': ma_mae,  '예측값 평균': ma_forecast.mean()},
    {'모델': 'Holt-Winters',                'MAPE(%)': hw_mape,  'MAE(개)': hw_mae,  '예측값 평균': hw_forecast.mean()},
    {'모델': 'XGBoost',                     'MAPE(%)': xgb_mape, 'MAE(개)': xgb_mae, '예측값 평균': xgb_forecast.mean()},
])
results['실제값 평균'] = test.mean()
best_model = results.loc[results['MAPE(%)'].idxmin(), '모델']

print(f'\n🏆 최우수 모델: {best_model}')
print()
results.style.highlight_min(subset=['MAPE(%)'], color='#D1FAE5').format({
    'MAPE(%)': '{:.1f}%', 'MAE(개)': '{:.0f}', '예측값 평균': '{:.0f}', '실제값 평균': '{:.0f}'
})

In [ ]:
# ── 예측값 시각화 ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'{target_name} 수요 예측 결과', fontsize=14, fontweight='bold')

# 시계열 비교
ax1.plot(train.index, train.values, color='#4B5563', linewidth=2, label='실제 (학습)')
ax1.plot(test.index, test.values, color='#111827', linewidth=2.5, marker='o', markersize=8, label='실제 (테스트)')
ax1.plot(test.index, ma_forecast.values, color='#F59E0B', linewidth=2, linestyle='--', marker='s', label=f'MA ({window}M)')
ax1.plot(test.index, hw_forecast.values, color='#10B981', linewidth=2, linestyle='--', marker='^', label='Holt-Winters')
ax1.plot(test.index, xgb_forecast.values, color='#4F46E5', linewidth=2, linestyle='--', marker='D', label='XGBoost')
ax1.axvline(x=test.index[0], color='red', linestyle=':', alpha=0.7, label='예측 시작')
ax1.set_title('시계열 예측 비교')
ax1.set_ylabel('판매량 (개)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.tick_params(axis='x', rotation=30)

# MAPE 바 차트
colors_bar = ['#F59E0B' if m != best_model else '#4F46E5' for m in results['모델']]
bars = ax2.barh(results['모델'], results['MAPE(%)'], color=colors_bar, alpha=0.85, height=0.5)
for bar, val in zip(bars, results['MAPE(%)']):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontweight='bold')
ax2.set_title('MAPE 비교 (낮을수록 좋음)')
ax2.set_xlabel('MAPE (%)')
ax2.axvline(x=10, color='green', linestyle='--', alpha=0.5, label='목표: 10% 이하')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_result.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. XGBoost 피처 중요도 분석

> 어떤 변수가 수요 예측에 가장 중요한가?

In [ ]:
feat_importance = pd.DataFrame({
    '피처': X_cols,
    '중요도': xgb_model.feature_importances_
}).sort_values('중요도', ascending=True)

label_map = {
    'month': '월 (계절성)',
    'quarter': '분기',
    'lag_1': '전월 판매량',
    'lag_3': '3개월 전 판매량',
    'lag_12': '전년 동월 판매량',
    'rolling_3': '3개월 이동평균',
    'rolling_6': '6개월 이동평균',
}
feat_importance['피처명'] = feat_importance['피처'].map(label_map)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(feat_importance['피처명'], feat_importance['중요도'], color='#4F46E5', alpha=0.8)
for bar, val in zip(bars, feat_importance['중요도']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_title('XGBoost 피처 중요도', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

top_feat = feat_importance.iloc[-1]['피처명']
print(f'\n💡 핵심 인사이트: 가장 중요한 예측 변수 = "{top_feat}"')

---
## 7. LLM 리포트 자동 생성

> **해결하는 문제**: 분석 결과를 매번 글로 정리하는 수작업 → LLM이 자동 요약

**🔑 OpenAI API Key 설정 방법**:
- Colab: `os.environ['OPENAI_API_KEY'] = 'sk-...'`
- 로컬: `.env` 파일에 `OPENAI_API_KEY=sk-...` 저장

In [ ]:
# API Key 설정 (실제 키로 교체 필요)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

def generate_report_with_llm(analysis_summary: str) -> str:
    """LLM을 이용한 구매 분석 리포트 자동 생성"""
    if not OPENAI_API_KEY:
        # API Key 없을 때: 규칙 기반 리포트 생성 (데모용)
        return generate_report_rule_based(analysis_summary)
    
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    prompt = f"""당신은 구매/SCM 전문 분석가입니다.
아래 데이터 분석 결과를 바탕으로 구매 담당자를 위한 월간 리포트를 작성해주세요.

## 분석 결과
{analysis_summary}

## 요구사항
- 구매 담당자가 바로 의사결정에 활용할 수 있는 형식
- 핵심 인사이트 3가지
- 다음 달 발주 권고사항
- 리스크 요인 1가지
- 300자 이내"""

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=500
    )
    return response.choices[0].message.content


def generate_report_rule_based(summary: str) -> str:
    """API Key 없을 때 규칙 기반 리포트 (데모)"""
    best_mape_row = results.loc[results['MAPE(%)'].idxmin()]
    return f"""📋 [자동 생성 구매 분석 리포트]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ 핵심 인사이트
1. {target_name} 수요는 계절성이 뚜렷하며 여름(7~8월) 피크 패턴이 확인됩니다.
2. 3개 모델 중 {best_model}이 MAPE {best_mape_row['MAPE(%)']:.1f}%로 가장 정확합니다.
3. 전년 동월 판매량이 예측에 가장 중요한 변수로 작용합니다.

📦 다음 달 발주 권고
- 권고 발주량: {int(xgb_forecast.mean() * 1.05):,}개 (예측값 + 5% 안전재고)
- 적용 모델: {best_model}
- 예측 신뢰도: {'높음' if best_mape_row['MAPE(%)'] < 10 else '보통'} (MAPE {best_mape_row['MAPE(%)']:.1f}%)

⚠️  리스크 요인
- 학습 데이터 24개월로 외부 충격(공급망 이슈 등) 반영 미흡
- 데이터 축적 시 Prophet 모델 도입 권장

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[자동 생성 | {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}]"""


# ── 분석 요약 데이터 구성 ────────────────────────────────────
analysis_summary = f"""
대상 상품: {target_name}
분석 기간: 2023-01 ~ 2024-12 (24개월)
월 평균 판매량: {ts.mean():.0f}개
최고 판매월: {ts.idxmax().strftime('%Y-%m')} ({ts.max():.0f}개)
최저 판매월: {ts.idxmin().strftime('%Y-%m')} ({ts.min():.0f}개)

[모델 성능 비교]
{results[['모델','MAPE(%)','MAE(개)']].to_string(index=False)}

최우수 모델: {best_model}
11월 예측값: {xgb_forecast.iloc[0]:.0f}개 (실제: {y_test.iloc[0]:.0f}개)
12월 예측값: {xgb_forecast.iloc[1]:.0f}개 (실제: {y_test.iloc[1]:.0f}개)
"""

# ── 리포트 생성 ───────────────────────────────────────────────
report = generate_report_with_llm(analysis_summary)
print(report)

# 리포트 저장
with open('monthly_report_auto.txt', 'w', encoding='utf-8') as f:
    f.write(report)
print('\n✅ monthly_report_auto.txt 저장 완료')

---
## 8. 전체 상품 일괄 예측 (확장 실습)

> 1개 상품 → 6개 전체 상품 자동 예측으로 확장

In [ ]:
all_results = []

for pid, meta in products.items():
    ts_p = raw[raw['상품코드'] == pid].set_index('날짜')['판매량'].sort_index()
    train_p = ts_p[:'2024-10']
    test_p  = ts_p['2024-11':]

    # Holt-Winters
    try:
        hw = ExponentialSmoothing(train_p, trend='add', seasonal='mul', seasonal_periods=12).fit(optimized=True)
        hw_pred = hw.forecast(len(test_p))
        hw_pred.index = test_p.index
        mape_val = mean_absolute_percentage_error(test_p, hw_pred) * 100
    except Exception:
        mape_val = np.nan
        hw_pred = pd.Series([train_p.mean()] * len(test_p), index=test_p.index)

    all_results.append({
        '상품코드': pid,
        '상품명': meta['name'],
        '카테고리': meta['category'],
        '11월_실제': int(test_p.iloc[0]),
        '12월_실제': int(test_p.iloc[1]),
        '11월_예측': int(hw_pred.iloc[0]),
        '12월_예측': int(hw_pred.iloc[1]),
        'MAPE(%)': round(mape_val, 1),
        '발주권고_12월': int(hw_pred.iloc[1] * 1.05),
    })

bulk_results = pd.DataFrame(all_results)

# 발주금액 환산
bulk_results['발주금액_권고(원)'] = bulk_results.apply(
    lambda r: r['발주권고_12월'] * products[r['상품코드']]['unit_price'], axis=1
)

print('📋 전체 상품 수요 예측 + 발주 권고 (Holt-Winters 기준)')
bulk_results.style \
    .background_gradient(subset=['MAPE(%)'], cmap='RdYlGn_r') \
    .format({'MAPE(%)': '{:.1f}%', '발주금액_권고(원)': '{:,.0f}'})

In [ ]:
# 발주 권고 최종 요약
total_order = bulk_results['발주금액_권고(원)'].sum()
avg_mape = bulk_results['MAPE(%)'].mean()

print('━' * 50)
print('📦 12월 자동 발주 권고 요약')
print('━' * 50)
for _, row in bulk_results.iterrows():
    print(f"  {row['상품명']:<20} {row['발주권고_12월']:>5}개  ({row['발주금액_권고(원)']:>10,.0f}원)")
print('━' * 50)
print(f"  총 발주 예상금액: {total_order:>15,.0f}원")
print(f"  평균 예측 정확도: {100-avg_mape:.1f}% (MAPE {avg_mape:.1f}%)")
print('━' * 50)
print()
print('💡 다음 단계 (AX 로드맵):')
print('  [현재] Holt-Winters 월별 예측')
print('  [Next] XGBoost + 외부 변수 (날씨, 이벤트) 추가')
print('  [Goal] 실시간 ERP 연동 → 자동 발주 시스템')

---
## 9. 실습 정리 & 다음 단계

### 오늘 해결한 것

| Before (수작업) | After (자동화) |
|-----------------|----------------|
| 엑셀 수동 정리 | `validate_data()` + 자동 시트 생성 |
| 경험 기반 발주 | Holt-Winters / XGBoost 모델 예측 |
| 정확도 미측정 | MAPE 정량 평가 |
| 수작업 리포트 | LLM 자동 요약 생성 |

### KPI 측정 방법
```python
# Forecast Accuracy = 100 - MAPE
accuracy = 100 - mean_absolute_percentage_error(actual, predicted) * 100
print(f'예측 정확도: {accuracy:.1f}%')  # 목표: 90% 이상
```

### 다음 실습 로드맵

```
Step 1 (완료) ─ 샘플 데이터 + MA / Holt-Winters / XGBoost
Step 2 (다음) ─ 실제 엑셀 연동 + Prophet 모델 추가
Step 3 (이후) ─ 외부 변수 (날씨 API, 공휴일) 피처 추가
Step 4 (목표) ─ ERP/SCM 연동 → 자동 발주 에이전트
```

### 생성된 파일
- `purchase_data_sample.xlsx` — 원본 샘플 데이터
- `purchase_report_auto.xlsx` — 자동 생성 리포트 (3개 시트)
- `eda_dashboard.png` — EDA 시각화
- `forecast_result.png` — 예측 결과 비교
- `monthly_report_auto.txt` — LLM 자동 리포트